In [2]:
import pickle
import csv
import os
import re
import nltk
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# NLTK resources
nltk.download('punkt')
nltk.download('wordnet')

# Load Loughran–McDonald dictionary
lm_dict = {}
with open("./data/text/Loughran-McDonald_MasterDictionary_1993-2024.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        word = row["Word"].lower()
        lm_dict[word] = {
            "positive": int(row["Positive"]),
            "negative": int(row["Negative"]),
            "uncertainty": int(row["Uncertainty"]),
            "litigious": int(row["Litigious"]),
            "strong_modal": int(row["Strong_Modal"]),
            "weak_modal": int(row["Weak_Modal"]),
            "constraining": int(row["Constraining"])
        }

lemmatizer = WordNetLemmatizer()

def preprocess_text(text, lemmatize=True):
    """Preprocess text without stopword removal."""
    text = text.lower()
    text = re.sub(r'\d+', '', text)   # remove numbers
    text = re.sub(r'\W+', ' ', text)  # remove punctuation
    tokens = word_tokenize(text)
    if lemmatize:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

def compute_ratios(tokens):
    """Compute LM dictionary ratios for a given tokenized document."""
    counts = {k: 0 for k in ["positive","negative","uncertainty","litigious","strong_modal","weak_modal","constraining"]}
    total = len(tokens) if len(tokens) > 0 else 1  # avoid div by zero

    for t in tokens:
        if t in lm_dict:
            for cat in counts:
                counts[cat] += lm_dict[t][cat]
    
    # Convert to ratios
    ratios = {f"{cat}_ratio": counts[cat] / total for cat in counts}
    return ratios

from tqdm import tqdm

# Collect results across years
all_years = []

for year in range(2005, 2026):  # adjust as needed
    file_path = f"./data/text/text_us_{year}.pkl"
    if not os.path.exists(file_path):
        print(f"Skipping {year}, file not found")
        continue

    df_year = pd.read_pickle(file_path)  # load yearly dataframe
    print(f"Processing year {year}, {len(df_year)} rows...")

    rows = []
    for _, row in tqdm(df_year.iterrows(), total=len(df_year), desc=f"Year {year}"):
        gvkey = row["gvkey"]
        yr = row["year"]

        # combine rf and mgmt text (skip if both empty)
        combined_text = ""
        if pd.notna(row["rf"]):
            combined_text += str(row["rf"]) + " "
        if pd.notna(row["mgmt"]):
            combined_text += str(row["mgmt"])

        if combined_text.strip() == "":
            continue  # skip rows with no text at all

        tokens = preprocess_text(combined_text, lemmatize=True)
        ratios = compute_ratios(tokens)

        entry = {"gvkey": gvkey, "year": yr}
        entry.update(ratios)
        rows.append(entry)

    all_years.extend(rows)
    print(f"Finished year {year}, collected {len(rows)} rows")

# Final combined dataframe
df_all = pd.DataFrame(all_years)
print(df_all.head())



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\shoai\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\shoai\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Processing year 2005, 16857 rows...


Year 2005: 100%|██████████| 16857/16857 [13:52<00:00, 20.25it/s]


Finished year 2005, collected 16825 rows
Processing year 2006, 16553 rows...


Year 2006: 100%|██████████| 16553/16553 [16:27<00:00, 16.77it/s]


...

Finished year 2023, collected 17793 rows
Processing year 2024, 20352 rows...


Year 2024: 100%|██████████| 20352/20352 [24:00<00:00, 14.13it/s] 


Finished year 2024, collected 20312 rows
Processing year 2025, 11644 rows...


Year 2025: 100%|██████████| 11644/11644 [15:59<00:00, 12.13it/s]


Finished year 2025, collected 11623 rows
      gvkey  year  positive_ratio  negative_ratio  uncertainty_ratio  \
0    6831.0  2005        0.000000        0.000000           0.000000   
1   11872.0  2005       13.130719       26.261438           0.000000   
2   24783.0  2005        0.000000        0.000000          32.934426   
3   61721.0  2005       13.852022       35.858856          41.608996   
4  146117.0  2005       14.754058       13.526799          37.534150   

   litigious_ratio  strong_modal_ratio  weak_modal_ratio  constraining_ratio  
0         0.000000            0.000000          0.000000            0.000000  
1         0.000000           13.130719          0.000000            0.000000  
2         0.000000            0.000000         32.934426           32.934426  
3        14.009763            6.164296         23.956695           15.975035  
4         5.537825            6.768453          7.999081           12.308729  


In [3]:
df_all.to_csv("lm_sentiment_2005_2025.csv", index=False)
print("Saved CSV: finbert_sentiment_rf_mgmt_2005_2025.csv")

Saved CSV: finbert_sentiment_rf_mgmt_2005_2025.csv


In [4]:
df_all

,gvkey,year,positive_ratio,negative_ratio,uncertainty_ratio,litigious_ratio,strong_modal_ratio,weak_modal_ratio,constraining_ratio
0,6831.0,2005,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,11872.0,2005,13.130719,26.261438,0.000000,0.000000,13.130719,0.000000,0.000000
2,24783.0,2005,0.000000,0.000000,32.934426,0.000000,0.000000,32.934426,32.934426
3,61721.0,2005,13.852022,35.858856,41.608996,14.009763,6.164296,23.956695,15.975035
4,146117.0,2005,14.754058,13.526799,37.534150,5.537825,6.768453,7.999081,12.308729
...,...,...,...,...,...,...,...,...,...
357975,38646.0,2025,0.000000,0.000000,60.098291,0.000000,0.000000,0.000000,0.000000
357976,NaN,2025,5.054768,18.670607,35.937195,6.174232,9.657132,17.889441,11.877147
357977,NaN,2025,9.910119,17.347799,26.445084,12.809338,9.916907,11.569724,12.810572
357978,NaN,2025,8.038400,12.831200,51.430400,12.857600,8.036000,17.679200,20.893600


In [ ]:
import pandas as pd

df1 = pd.read_csv("finbert_sentiment_rf_mgmt_2005_2025.csv")
df2 = pd.read_csv("lm_sentiment_2005_2025.csv")

merged = pd.merge(df1, df2, on=["gvkey", "year"], how="inner")

merged.to_csv("text_data_sentiment.csv", index=False)

In [7]:
merged

,bearish,neutral,bullish,gvkey,year,positive_ratio,negative_ratio,uncertainty_ratio,litigious_ratio,strong_modal_ratio,weak_modal_ratio,constraining_ratio
0,0.010647,0.932460,0.056893,6831.0,2005,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.010647,0.932460,0.056893,6831.0,2005,14.336475,36.107684,20.084262,9.181377,1.721508,8.607541,15.501571
2,0.010647,0.932460,0.056893,6831.0,2005,11.462977,32.845934,27.141595,8.351260,2.087815,15.136659,17.232788
3,0.010647,0.932460,0.056893,6831.0,2005,15.449145,88.430342,67.825214,33.483333,4.292735,44.644444,18.893162
4,0.010647,0.932460,0.056893,6831.0,2005,1.812200,27.970756,30.752342,9.638794,4.589902,8.261823,16.989947
...,...,...,...,...,...,...,...,...,...,...,...,...
14642883,0.216606,0.662318,0.121076,1585.0,2025,10.050104,50.244623,42.407599,22.404015,9.761749,23.364187,18.411582
14642884,0.071191,0.834922,0.093886,116025.0,2025,11.332243,18.494871,22.995838,19.712353,8.063476,5.375650,11.649026
14642885,0.254874,0.622000,0.123126,28378.0,2025,5.413848,48.828519,36.801785,24.770432,4.403632,23.040434,20.056482
14642886,0.363818,0.513946,0.122235,36152.0,2025,6.493233,48.803657,43.238529,16.844458,4.509540,24.271935,17.378557
